# Module 2.2: 注意力机制 (Attention Mechanism)

## 1. 本章概览

### 📚 学习目标

1. **动机**：理解RNN的局限性和注意力机制的必要性
2. **基础注意力**：实现简单的注意力机制
3. **注意力变体**：对比不同的注意力计算方法
4. **缩放点积注意力**：理解Transformer的核心组件

### 🎯 核心问题

- 为什么RNN难以处理长序列？
- 注意力机制如何解决这个问题？
- 不同注意力机制有什么区别？

### ⏱️ 预计学习时间：3-4小时

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported")

## 2. 动机与背景

### RNN的局限性

**问题**：在序列到序列任务中（如机器翻译），RNN将整个输入序列压缩成一个固定长度的向量（context vector）。

```
输入: "The cat sat on the mat"
编码器: [word1, word2, ..., word6] → [固定长度向量]
解码器: [固定长度向量] → [翻译结果]
```

**信息瓶颈 (Information Bottleneck)** 是指模型被迫将任意长度的输入压缩到固定维度的向量中，导致信息不可避免地丢失。在本章场景中，它描述了 RNN 编码器将整个序列压缩为单一上下文向量时面临的核心困难。

**挑战**：
- 📉 **信息瓶颈**：长序列的信息难以压缩到单一向量
- 🔄 **梯度消失**：长距离依赖难以学习
- ⏱️ **顺序处理**：无法并行化

### 注意力机制的解决方案

**核心思想**：不是将所有信息压缩到一个向量，而是让解码器在每一步**直接访问**所有编码器状态，并**动态选择**关注哪些部分。

```
编码器状态: [h1, h2, h3, h4, h5, h6]
解码器步骤1: 关注 [h1=0.6, h2=0.3, h3=0.1, ...] → 生成第一个词
解码器步骤2: 关注 [h1=0.1, h2=0.1, h3=0.7, ...] → 生成第二个词
```

**优势**：
- ✅ 直接访问所有输入信息
- ✅ 可解释性：可视化注意力权重
- ✅ 更好的长距离依赖建模

## 3. 基础注意力机制

### 3.1 注意力的三个步骤

**输入**：
- Query (查询): $q \in \mathbb{R}^{d_q}$ - 当前解码器状态
- Keys (键): $K \in \mathbb{R}^{n \times d_k}$ - 所有编码器状态
- Values (值): $V \in \mathbb{R}^{n \times d_v}$ - 所有编码器状态（通常 $V = K$）

**步骤**：

1. **计算注意力分数** (Attention Scores):
   $$\text{scores} = \text{similarity}(q, K)$$

2. **归一化为权重** (Attention Weights):

   **Softmax 函数 (Softmax Function)** 是将任意实数向量归一化为概率分布的函数，定义为 $\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$，输出值在 $(0,1)$ 之间且总和为 1。在本章场景中，它用于将注意力分数转换为注意力权重（概率分布）。

   $$\alpha = \text{softmax}(\text{scores})$$

3. **加权求和** (Context Vector):
   $$\text{context} = \sum_{i=1}^{n} \alpha_i \cdot v_i$$

**直觉**：注意力权重 $\alpha$ 告诉我们应该关注输入的哪些部分。

In [ ]:
# 🔬 Micro Practice: Basic Attention Mechanism
# Goal: Implement simple attention from scratch
# Expected outcome: Understand the three steps of attention

def basic_attention(query, keys, values):
    """
    Basic attention mechanism using dot product
    
    Args:
        query: (d_q,) - current decoder state
        keys: (n, d_k) - all encoder states
        values: (n, d_v) - all encoder states
    
    Returns:
        context: (d_v,) - weighted sum of values
        attention_weights: (n,) - attention distribution
    """
    # Step 1: Compute attention scores (dot product)
    scores = np.dot(keys, query)  # (n,)
    
    # Step 2: Normalize to attention weights (softmax)
    attention_weights = np.exp(scores) / np.sum(np.exp(scores))
    
    # Step 3: Weighted sum of values
    context = np.dot(attention_weights, values)  # (d_v,)
    
    return context, attention_weights

# Toy example: Translate "I love AI"
# Encoder states (3 words, 4-dim embeddings)
encoder_states = np.array([
    [1.0, 0.5, 0.2, 0.1],  # "I"
    [0.3, 0.8, 0.6, 0.4],  # "love"
    [0.2, 0.3, 0.9, 0.7],  # "AI"
])

# Current decoder state (generating first word of translation)
decoder_state = np.array([0.5, 0.6, 0.7, 0.3])

# Apply attention
context, weights = basic_attention(decoder_state, encoder_states, encoder_states)

print("Input words: ['I', 'love', 'AI']")
print(f"\nDecoder state: {decoder_state}")
print(f"\nAttention weights:")
for i, (word, weight) in enumerate(zip(['I', 'love', 'AI'], weights)):
    print(f"  {word}: {weight:.3f} {'█' * int(weight * 50)}")

print(f"\nContext vector: {context}")
print("\n✓ Basic attention computed!")

In [ ]:
# 🔬 Micro Practice: Visualize Attention Weights
# Goal: See which parts of input the model attends to

# Simulate attention for multiple decoder steps
input_words = ['I', 'love', 'AI']
output_words = ['我', '爱', '人工智能']

# Simulate different decoder states
decoder_states = [
    np.array([0.9, 0.2, 0.1, 0.1]),  # Focus on "I"
    np.array([0.2, 0.9, 0.5, 0.3]),  # Focus on "love"
    np.array([0.1, 0.3, 0.9, 0.8]),  # Focus on "AI"
]

# Compute attention for each step
attention_matrix = []
for decoder_state in decoder_states:
    _, weights = basic_attention(decoder_state, encoder_states, encoder_states)
    attention_matrix.append(weights)

attention_matrix = np.array(attention_matrix)

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(attention_matrix, 
            xticklabels=input_words,
            yticklabels=output_words,
            annot=True, fmt='.2f',
            cmap='YlOrRd',
            cbar_kws={'label': 'Attention Weight'})
plt.xlabel('Input (English)', fontsize=12)
plt.ylabel('Output (Chinese)', fontsize=12)
plt.title('Attention Weights Heatmap', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

print("✓ Attention visualization shows alignment between input and output!")

## 4. 注意力机制的变体

### 4.1 三种主要的注意力机制

不同的注意力机制主要区别在于如何计算**注意力分数**（相似度函数）。

#### 1. **加性注意力** (Additive Attention / Bahdanau Attention)

$$\text{score}(q, k_i) = v^T \tanh(W_q q + W_k k_i)$$

- 使用可学习的权重矩阵 $W_q, W_k$ 和向量 $v$
- 通过 tanh 激活函数引入非线性
- **复杂度**: $O(d_q \cdot d_k)$

#### 2. **乘性注意力** (Multiplicative Attention / Luong Attention)

$$\text{score}(q, k_i) = q^T W k_i$$

- 使用可学习的权重矩阵 $W$
- 比加性注意力更简单
- **复杂度**: $O(d_q \cdot d_k)$

#### 3. **缩放点积注意力** (Scaled Dot-Product Attention)

$$\text{score}(q, k_i) = \frac{q^T k_i}{\sqrt{d_k}}$$

- 不需要额外的可学习参数
- 除以 $\sqrt{d_k}$ 防止梯度消失
- **复杂度**: $O(d_k)$ - 最快！
- **Transformer 使用的方法**

### 4.2 为什么需要缩放？

当 $d_k$ 很大时，点积的值会变得很大，导致 softmax 函数进入饱和区，梯度变得很小。

**示例**：假设 $q$ 和 $k$ 的每个元素都是独立的标准正态分布 $\mathcal{N}(0, 1)$

- 点积 $q^T k$ 的方差为 $d_k$
- 当 $d_k = 64$ 时，点积可能在 $[-20, 20]$ 范围
- Softmax 在这个范围内几乎是 one-hot，梯度接近 0

**解决方案**：除以 $\sqrt{d_k}$ 将方差归一化到 1

In [ ]:
# 🔬 Micro Practice: Implement Attention Variants
# Goal: Compare different attention mechanisms
# Expected outcome: Understand trade-offs between variants

class AdditiveAttention:
    """Bahdanau Attention"""
    
    def __init__(self, d_q, d_k, d_hidden):
        self.W_q = np.random.randn(d_q, d_hidden) * 0.1
        self.W_k = np.random.randn(d_k, d_hidden) * 0.1
        self.v = np.random.randn(d_hidden) * 0.1
    
    def __call__(self, query, keys, values):
        """
        Args:
            query: (d_q,)
            keys: (n, d_k)
            values: (n, d_v)
        """
        # Transform query and keys
        q_transformed = np.dot(query, self.W_q)  # (d_hidden,)
        k_transformed = np.dot(keys, self.W_k)   # (n, d_hidden)
        
        # Additive: v^T * tanh(W_q*q + W_k*k)
        scores = np.dot(np.tanh(q_transformed + k_transformed), self.v)  # (n,)
        
        # Softmax
        attention_weights = np.exp(scores) / np.sum(np.exp(scores))
        
        # Weighted sum
        context = np.dot(attention_weights, values)
        
        return context, attention_weights

class MultiplicativeAttention:
    """Luong Attention"""
    
    def __init__(self, d_q, d_k):
        self.W = np.random.randn(d_q, d_k) * 0.1
    
    def __call__(self, query, keys, values):
        """
        Args:
            query: (d_q,)
            keys: (n, d_k)
            values: (n, d_v)
        """
        # Multiplicative: q^T * W * k
        q_transformed = np.dot(query, self.W)  # (d_k,)
        scores = np.dot(keys, q_transformed)   # (n,)
        
        # Softmax
        attention_weights = np.exp(scores) / np.sum(np.exp(scores))
        
        # Weighted sum
        context = np.dot(attention_weights, values)
        
        return context, attention_weights

class ScaledDotProductAttention:
    """Transformer Attention"""
    
    def __call__(self, query, keys, values):
        """
        Args:
            query: (d_k,)
            keys: (n, d_k)
            values: (n, d_v)
        """
        d_k = keys.shape[1]
        
        # Scaled dot product: (q^T * k) / sqrt(d_k)
        scores = np.dot(keys, query) / np.sqrt(d_k)  # (n,)
        
        # Softmax
        attention_weights = np.exp(scores) / np.sum(np.exp(scores))
        
        # Weighted sum
        context = np.dot(attention_weights, values)
        
        return context, attention_weights

# Test all three variants
d_q, d_k, d_v = 4, 4, 4
n_keys = 3

query = np.random.randn(d_q)
keys = np.random.randn(n_keys, d_k)
values = np.random.randn(n_keys, d_v)

# Initialize attention mechanisms
additive_attn = AdditiveAttention(d_q, d_k, d_hidden=8)
multiplicative_attn = MultiplicativeAttention(d_q, d_k)
scaled_dot_attn = ScaledDotProductAttention()

# Compute attention
_, weights_additive = additive_attn(query, keys, values)
_, weights_multiplicative = multiplicative_attn(query, keys, values)
_, weights_scaled = scaled_dot_attn(query, keys, values)

print("Attention Weights Comparison:\\n")
print(f"{'Mechanism':<25} {'Key 1':<10} {'Key 2':<10} {'Key 3':<10}")
print("-" * 55)
print(f"{'Additive (Bahdanau)':<25} {weights_additive[0]:.4f}     {weights_additive[1]:.4f}     {weights_additive[2]:.4f}")
print(f"{'Multiplicative (Luong)':<25} {weights_multiplicative[0]:.4f}     {weights_multiplicative[1]:.4f}     {weights_multiplicative[2]:.4f}")
print(f"{'Scaled Dot-Product':<25} {weights_scaled[0]:.4f}     {weights_scaled[1]:.4f}     {weights_scaled[2]:.4f}")

print("\\n✓ All three attention variants implemented!")

In [ ]:
# 🔬 Micro Practice: Visualize Attention Differences
# Goal: Compare attention patterns from different mechanisms

# Create a more interesting example
input_sentence = ['The', 'cat', 'sat', 'on', 'the', 'mat']
n_words = len(input_sentence)

# Create encoder states (simulated)
np.random.seed(42)
encoder_states = np.random.randn(n_words, 4)

# Create different queries (simulating different decoder steps)
queries = [
    np.array([1.0, 0.2, 0.1, 0.1]),  # Query 1
    np.array([0.2, 1.0, 0.3, 0.2]),  # Query 2
    np.array([0.1, 0.3, 1.0, 0.5]),  # Query 3
]

# Compute attention for all mechanisms
attention_patterns = {
    'Additive': [],
    'Multiplicative': [],
    'Scaled Dot-Product': []
}

additive = AdditiveAttention(4, 4, 8)
multiplicative = MultiplicativeAttention(4, 4)
scaled_dot = ScaledDotProductAttention()

for query in queries:
    _, w1 = additive(query, encoder_states, encoder_states)
    _, w2 = multiplicative(query, encoder_states, encoder_states)
    _, w3 = scaled_dot(query, encoder_states, encoder_states)
    
    attention_patterns['Additive'].append(w1)
    attention_patterns['Multiplicative'].append(w2)
    attention_patterns['Scaled Dot-Product'].append(w3)

# Visualize side-by-side
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, patterns) in enumerate(attention_patterns.items()):
    patterns_array = np.array(patterns)
    
    sns.heatmap(patterns_array, 
                xticklabels=input_sentence,
                yticklabels=[f'Query {i+1}' for i in range(3)],
                annot=True, fmt='.2f',
                cmap='YlOrRd',
                ax=axes[idx],
                cbar_kws={'label': 'Weight'})
    
    axes[idx].set_title(f'{name}\\nAttention', fontsize=12, weight='bold')
    axes[idx].set_xlabel('Input Words')
    axes[idx].set_ylabel('Decoder Steps')

plt.tight_layout()
plt.show()

print("✓ Different attention mechanisms produce different attention patterns!")

## 5. 缩放点积注意力 (Scaled Dot-Product Attention)

### 5.1 完整公式

这是 Transformer 中使用的注意力机制：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**输入**：
- $Q \in \mathbb{R}^{n_q \times d_k}$: 查询矩阵 (n_q 个查询)
- $K \in \mathbb{R}^{n_k \times d_k}$: 键矩阵 (n_k 个键)
- $V \in \mathbb{R}^{n_k \times d_v}$: 值矩阵 (n_k 个值)

**输出**：
- $\text{Output} \in \mathbb{R}^{n_q \times d_v}$: 每个查询的上下文向量

**步骤**：
1. 计算所有查询和键的点积：$QK^T \in \mathbb{R}^{n_q \times n_k}$
2. 缩放：除以 $\sqrt{d_k}$
3. Softmax：每一行归一化为概率分布
4. 加权求和：与值矩阵相乘

### 5.2 关键特性

**1. 批量计算**：一次处理多个查询
**2. 可并行化**：所有计算都是矩阵运算
**3. 掩码支持**：可以屏蔽某些位置（用于自回归生成）

### 5.3 掩码 (Masking)

在某些情况下，我们需要防止注意力关注某些位置：

- **Padding mask**: 忽略填充的位置
- **Look-ahead mask**: 在解码时，防止关注未来的位置

实现方法：在 softmax 之前，将不应该关注的位置设为 $-\infty$

In [ ]:
# 🔬 Micro Practice: Scaled Dot-Product Attention (NumPy)
# Goal: Implement complete scaled dot-product attention with masking
# Expected outcome: Understand batched attention computation

def scaled_dot_product_attention_numpy(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    
    Args:
        Q: (n_q, d_k) - queries
        K: (n_k, d_k) - keys
        V: (n_k, d_v) - values
        mask: (n_q, n_k) - optional mask (1 = keep, 0 = mask)
    
    Returns:
        output: (n_q, d_v) - attention output
        attention_weights: (n_q, n_k) - attention distribution
    """
    d_k = K.shape[1]
    
    # Step 1: Compute attention scores
    scores = np.matmul(Q, K.T) / np.sqrt(d_k)  # (n_q, n_k)
    
    # Step 2: Apply mask (optional)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    
    # Step 3: Softmax
    attention_weights = np.exp(scores) / np.sum(np.exp(scores), axis=-1, keepdims=True)
    
    # Step 4: Weighted sum
    output = np.matmul(attention_weights, V)  # (n_q, d_v)
    
    return output, attention_weights

# Example: Self-attention on a sequence
seq_length = 5
d_model = 8

# Create Q, K, V (in self-attention, they're all from the same input)
np.random.seed(42)
X = np.random.randn(seq_length, d_model)
Q = K = V = X

# Compute attention
output, attn_weights = scaled_dot_product_attention_numpy(Q, K, V)

print(f"Input shape: {X.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"\\nAttention weights (each row sums to 1):")
print(attn_weights)
print(f"\\nRow sums: {attn_weights.sum(axis=1)}")

# Visualize attention pattern
plt.figure(figsize=(8, 6))
sns.heatmap(attn_weights, 
            annot=True, fmt='.2f',
            cmap='YlOrRd',
            xticklabels=[f'Pos {i}' for i in range(seq_length)],
            yticklabels=[f'Pos {i}' for i in range(seq_length)],
            cbar_kws={'label': 'Attention Weight'})
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Self-Attention Pattern', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

print("\\n✓ Scaled dot-product attention implemented!")

In [ ]:
# 🔬 Micro Practice: Masking Example
# Goal: Understand how masking works in attention
# Expected outcome: See masked attention patterns

# Create a look-ahead mask (for autoregressive generation)
def create_look_ahead_mask(size):
    """
    Create mask that prevents attending to future positions
    
    Returns:
        mask: (size, size) - lower triangular matrix
    """
    mask = np.tril(np.ones((size, size)))
    return mask

# Example with masking
seq_length = 5
mask = create_look_ahead_mask(seq_length)

print("Look-ahead mask (1 = can attend, 0 = masked):")
print(mask.astype(int))

# Apply masked attention
output_masked, attn_weights_masked = scaled_dot_product_attention_numpy(Q, K, V, mask=mask)

# Visualize masked vs unmasked
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Unmasked
sns.heatmap(attn_weights, 
            annot=True, fmt='.2f',
            cmap='YlOrRd',
            xticklabels=[f'Pos {i}' for i in range(seq_length)],
            yticklabels=[f'Pos {i}' for i in range(seq_length)],
            ax=ax1,
            cbar_kws={'label': 'Weight'})
ax1.set_title('Unmasked Attention\\n(Bidirectional)', fontsize=12, weight='bold')
ax1.set_xlabel('Key Position')
ax1.set_ylabel('Query Position')

# Masked
sns.heatmap(attn_weights_masked, 
            annot=True, fmt='.2f',
            cmap='YlOrRd',
            xticklabels=[f'Pos {i}' for i in range(seq_length)],
            yticklabels=[f'Pos {i}' for i in range(seq_length)],
            ax=ax2,
            cbar_kws={'label': 'Weight'})
ax2.set_title('Masked Attention\\n(Causal/Autoregressive)', fontsize=12, weight='bold')
ax2.set_xlabel('Key Position')
ax2.set_ylabel('Query Position')

plt.tight_layout()
plt.show()

print("\\n✓ Masking prevents attending to future positions!")

In [ ]:
# 🔬 Micro Practice: PyTorch Implementation
# Goal: Implement attention using PyTorch for GPU acceleration
# Expected outcome: Production-ready attention module

class ScaledDotProductAttentionPyTorch(nn.Module):
    """
    Scaled Dot-Product Attention in PyTorch
    Supports batching and GPU acceleration
    """
    
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q: (batch, n_q, d_k)
            K: (batch, n_k, d_k)
            V: (batch, n_k, d_v)
            mask: (batch, n_q, n_k) or (n_q, n_k)
        
        Returns:
            output: (batch, n_q, d_v)
            attention_weights: (batch, n_q, n_k)
        """
        d_k = Q.size(-1)
        
        # Compute attention scores: Q @ K^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
        
        # Apply mask (set masked positions to -inf)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Softmax
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Weighted sum
        output = torch.matmul(attention_weights, V)
        
        return output, attention_weights

# Test PyTorch implementation
batch_size = 2
seq_length = 4
d_model = 8

# Create random inputs
Q = torch.randn(batch_size, seq_length, d_model)
K = torch.randn(batch_size, seq_length, d_model)
V = torch.randn(batch_size, seq_length, d_model)

# Create attention module
attention = ScaledDotProductAttentionPyTorch(dropout=0.0)

# Forward pass
output, attn_weights = attention(Q, K, V)

print(f"PyTorch Implementation:")
print(f"  Input Q shape: {Q.shape}")
print(f"  Output shape: {output.shape}")
print(f"  Attention weights shape: {attn_weights.shape}")
print(f"\\nFirst batch attention weights:")
print(attn_weights[0].detach().numpy())

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(attn_weights[0].detach().numpy(), 
            annot=True, fmt='.2f',
            cmap='YlOrRd',
            xticklabels=[f'K{i}' for i in range(seq_length)],
            yticklabels=[f'Q{i}' for i in range(seq_length)],
            cbar_kws={'label': 'Attention Weight'})
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('PyTorch Attention (Batch 0)', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

print("\\n✓ PyTorch implementation ready for production use!")

## 6. 实践示例：简单的序列到序列任务

让我们用注意力机制解决一个实际问题：将数字序列反转。

**任务**：输入 `[1, 2, 3, 4]` → 输出 `[4, 3, 2, 1]`

这个简单任务可以展示注意力如何学习对齐输入和输出位置。

In [ ]:
# 🔬 Micro Practice: Practical Example - Sequence Reversal
# Goal: See attention in action on a real task
# Expected outcome: Attention learns to align reversed positions

# Simulate a trained model's attention pattern for sequence reversal
# In reality, this would be learned through training
def simulate_reversal_attention(seq_length):
    """
    Simulate attention pattern for sequence reversal
    Position i in output should attend to position (n-1-i) in input
    """
    attention = np.zeros((seq_length, seq_length))
    for i in range(seq_length):
        # Output position i attends strongly to input position (n-1-i)
        target_pos = seq_length - 1 - i
        attention[i, target_pos] = 0.8
        # Small attention to other positions (noise)
        attention[i, :] += 0.2 / seq_length
    
    # Normalize
    attention = attention / attention.sum(axis=1, keepdims=True)
    return attention

# Example
seq_length = 6
input_seq = list(range(1, seq_length + 1))
output_seq = list(reversed(input_seq))

attention_pattern = simulate_reversal_attention(seq_length)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Attention heatmap
sns.heatmap(attention_pattern, 
            annot=True, fmt='.2f',
            cmap='YlOrRd',
            xticklabels=input_seq,
            yticklabels=output_seq,
            ax=ax1,
            cbar_kws={'label': 'Attention Weight'})
ax1.set_xlabel('Input Position', fontsize=12)
ax1.set_ylabel('Output Position', fontsize=12)
ax1.set_title('Attention Pattern for Sequence Reversal', fontsize=13, weight='bold')

# Alignment visualization
ax2.set_xlim(-0.5, seq_length - 0.5)
ax2.set_ylim(-0.5, seq_length - 0.5)

# Draw input and output positions
for i, val in enumerate(input_seq):
    ax2.scatter(i, seq_length - 1, s=200, c='lightblue', edgecolors='black', linewidth=2, zorder=3)
    ax2.text(i, seq_length - 1, str(val), ha='center', va='center', fontsize=12, weight='bold')

for i, val in enumerate(output_seq):
    ax2.scatter(i, 0, s=200, c='lightcoral', edgecolors='black', linewidth=2, zorder=3)
    ax2.text(i, 0, str(val), ha='center', va='center', fontsize=12, weight='bold')

# Draw attention connections
for out_pos in range(seq_length):
    for in_pos in range(seq_length):
        weight = attention_pattern[out_pos, in_pos]
        if weight > 0.3:  # Only draw strong connections
            ax2.plot([in_pos, out_pos], [seq_length - 1, 0], 
                    'r-', alpha=weight, linewidth=weight * 3)

ax2.set_xlabel('Position', fontsize=12)
ax2.set_yticks([0, seq_length - 1])
ax2.set_yticklabels(['Output', 'Input'])
ax2.set_title('Attention Alignment', fontsize=13, weight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Attention learns to align reversed positions!")

## 7. 常见问题与调试

### Q1: 为什么需要缩放因子 $\sqrt{d_k}$？

**A:** 当维度 $d_k$ 很大时，点积的方差会变大，导致 softmax 进入饱和区（输出接近 one-hot），梯度变得很小。缩放可以保持方差稳定。

**数学解释**：
- 假设 $q$ 和 $k$ 的元素独立同分布 $\sim \mathcal{N}(0, 1)$
- 点积 $q^T k = \sum_{i=1}^{d_k} q_i k_i$ 的方差为 $d_k$
- 除以 $\sqrt{d_k}$ 后，方差归一化为 1

### Q2: 注意力机制的计算复杂度是多少？

**A:** 对于序列长度 $n$ 和维度 $d$：
- **时间复杂度**: $O(n^2 \cdot d)$ - 主要来自 $QK^T$ 的矩阵乘法
- **空间复杂度**: $O(n^2)$ - 存储注意力权重矩阵

这是 Transformer 处理长序列的主要瓶颈。

### Q3: Self-Attention 和 Cross-Attention 有什么区别？

**A:**
- **Self-Attention**: $Q, K, V$ 都来自同一个序列（如 Transformer 编码器）
- **Cross-Attention**: $Q$ 来自一个序列，$K, V$ 来自另一个序列（如 Transformer 解码器）

### Q4: 如何处理变长序列？

**A:** 使用 **padding mask**：
1. 将短序列填充到批次中的最大长度
2. 创建 mask 标记哪些位置是填充的
3. 在计算注意力时，将填充位置的分数设为 $-\infty$

### Q5: 注意力权重可以用来做什么？

**A:**
- **可解释性**: 可视化模型关注输入的哪些部分
- **对齐**: 在翻译任务中，显示源语言和目标语言的对应关系
- **调试**: 检查模型是否学到了合理的模式

## 8. 总结与展望

### 核心要点

1. **动机**: 注意力机制解决了 RNN 的信息瓶颈问题
2. **机制**: 通过动态加权求和，让模型关注输入的相关部分
3. **变体**: 加性、乘性、缩放点积（Transformer 使用）
4. **关键技术**: 缩放、掩码、批量计算

### 注意力机制的演进

```
RNN (固定上下文向量)
  ↓
Bahdanau Attention (加性注意力, 2014)
  ↓
Luong Attention (乘性注意力, 2015)
  ↓
Scaled Dot-Product Attention (Transformer, 2017)
  ↓
Multi-Head Attention (下一章)
```

### 下一章预告

**Module 2.3: Multi-Head Attention**
- 为什么需要多头注意力？
- 如何实现并行的多个注意力头？
- Multi-Head Attention 在 Transformer 中的作用

### 💡 思考题

1. 为什么注意力机制比 RNN 更容易并行化？
2. 如果不使用缩放因子会发生什么？
3. 如何修改注意力机制来处理图结构数据？
4. 注意力机制的 $O(n^2)$ 复杂度如何优化？（提示：稀疏注意力、线性注意力）

## 思考题参考答案

### 问题 1：为什么注意力机制比 RNN 更容易并行化？

这是注意力机制相比 RNN 最根本的架构优势之一，根源在于**计算依赖关系**的差异。

**RNN 的顺序依赖（无法并行）：**

RNN 在时间步 $t$ 的计算依赖于前一步的隐藏状态 $h_{t-1}$：

$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$

这意味着 $h_2$ 必须等 $h_1$ 计算完毕，$h_3$ 必须等 $h_2$ 计算完毕……形成严格的**顺序链**，无法在序列维度上并行，训练时间与序列长度 $T$ 成正比。

**注意力机制的并行计算：**

缩放点积注意力的核心操作是矩阵乘法：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- $Q, K, V$ 的计算只依赖于**当前输入**（线性变换），各位置之间无依赖
- $QK^T$ 是一次矩阵乘法，GPU 可以对所有位置对同时计算
- 整个序列的注意力权重可以**一次性**计算出来

**直观对比：**

| 维度 | RNN | 注意力机制 |
|------|-----|-----------|
| 计算图 | 有向无环图（顺序链） | 全连接图（可并行） |
| 时间步依赖 | $h_t$ 依赖 $h_{t-1}$ | 无时间步依赖 |
| GPU 利用率 | 低（顺序执行） | 高（批量矩阵运算） |
| 训练速度 | 慢（序列越长越慢） | 快（与序列长度弱相关） |

**注意**：注意力机制在推理时，若需要自回归生成（如 GPT），仍然需要逐步生成，但可以通过 **KV Cache** 技术避免重复计算。

---

### 问题 2：如果不使用缩放因子会发生什么？

缩放因子 $\frac{1}{\sqrt{d_k}}$ 看似微小，但对训练稳定性至关重要。

**数学分析：**

假设 $q, k \in \mathbb{R}^{d_k}$，每个元素独立同分布 $\sim \mathcal{N}(0, 1)$，则：

$$q^T k = \sum_{i=1}^{d_k} q_i k_i$$

由于各项独立，$\text{Var}(q^T k) = d_k$，**标准差为 $\sqrt{d_k}$**。

当 $d_k$ 较大时（Transformer 中通常为 64 或更大），点积结果分布在 $[-\text{几十}, +\text{几十}]$ 的范围内。

**不缩放的后果：**

以 $d_k = 64$ 为例，设两个位置的分数分别为 $s_1 = 20$，$s_2 = -5$，softmax 结果为：

$$\alpha_1 = \frac{e^{20}}{e^{20} + e^{-5}} \approx 0.9999999\ldots$$

Softmax 几乎变成 **one-hot**（硬注意力），导致：
1. **梯度消失**：softmax 饱和区的梯度接近 0，参数无法有效更新
2. **注意力坍缩**：模型只关注一个位置，丧失了"软"注意力的优势
3. **训练不稳定**：梯度几乎为零，学习极其缓慢甚至停滞

**缩放后的效果：**

除以 $\sqrt{d_k}$ 后，$\text{Var}\left(\frac{q^T k}{\sqrt{d_k}}\right) = 1$，分数在 $[-3, 3]$ 范围内，softmax 输出更平滑，梯度正常流动。

---

### 问题 3：如何修改注意力机制来处理图结构数据？

标准注意力机制处理的是序列（全连接的注意力），而图数据具有明确的边结构。将注意力扩展到图的核心思想是：**只在存在边的节点对之间计算注意力**。

**方法一：图注意力网络（GAT，Graph Attention Network）**

对于节点 $i$，只对其邻居节点 $j \in \mathcal{N}(i)$ 计算注意力：

$$e_{ij} = \text{LeakyReLU}\left(\vec{a}^T [W h_i \| W h_j]\right)$$

$$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}(i)} \exp(e_{ik})}$$

$$h_i' = \sigma\left(\sum_{j \in \mathcal{N}(i)} \alpha_{ij} W h_j\right)$$

其中 $\|$ 表示拼接，$W$ 是可学习的线性变换，$\vec{a}$ 是注意力参数向量。

**方法二：稀疏注意力 + 邻接矩阵掩码**

在缩放点积注意力中，用图的邻接矩阵 $A$ 作为掩码：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

其中掩码矩阵：
$$M_{ij} = \begin{cases} 0 & \text{若 } (i,j) \in \text{边集} \\ -\infty & \text{若 } (i,j) \notin \text{边集} \end{cases}$$

**关键扩展点：**

| 扩展方向 | 具体方式 |
|---------|---------|
| 边特征融合 | 将边的特征嵌入加入注意力分数计算 |
| 方向信息 | 区分入边和出边，使用不同的权重矩阵 |
| 多跳连接 | 多层 GAT 隐式捕获多跳邻居信息 |
| 位置编码 | 用图上的拉普拉斯特征向量作为位置编码 |

**代表性应用**：Graphformer（分子属性预测）、Graph-BERT（图节点分类）。

---

### 问题 4：注意力机制的 $O(n^2)$ 复杂度如何优化？

注意力的 $O(n^2 \cdot d)$ 时间复杂度和 $O(n^2)$ 空间复杂度在长序列（$n > 2048$）时成为瓶颈。主要优化方向有以下几类：

**1. 稀疏注意力（Sparse Attention）**

不计算所有 $n^2$ 对，只计算部分重要的注意力：

- **局部窗口注意力**：每个 token 只关注左右 $w$ 个邻居，复杂度 $O(n \cdot w)$
  - 代表：Longformer、BigBird
- **滑动窗口 + 全局 token**：少数全局 token 可关注所有位置
- **随机注意力**：随机采样部分位置计算

**2. 线性注意力（Linear Attention）**

通过核函数近似 softmax，将注意力分解为线性操作：

$$\text{Attention}(Q, K, V) \approx \phi(Q) \left(\phi(K)^T V\right)$$

利用矩阵乘法结合律，复杂度从 $O(n^2 d)$ 降至 $O(n d^2)$。
- 代表：Linear Transformer、Performer（使用随机特征近似）

**3. 低秩近似**

- **Linformer**：将 $K, V$ 投影到低维（$k \ll n$），复杂度 $O(n \cdot k)$
- 假设注意力矩阵是低秩的，用 SVD 近似

**4. 分块计算（FlashAttention）**

不降低理论复杂度，但通过 **IO 感知的分块计算**（Tiling）避免将 $n \times n$ 注意力矩阵写回 HBM，大幅减少显存读写：
- 实际速度提升 2–4 倍，显存占用降至 $O(n)$
- 现已成为主流 LLM 训练的标配

**5. 混合架构**

在不同层使用不同的注意力范围：
- 底层：局部注意力（捕捉语法）
- 顶层：全局注意力（捕捉语义）

**复杂度对比：**

| 方法 | 时间复杂度 | 空间复杂度 | 备注 |
|------|-----------|-----------|------|
| 标准注意力 | $O(n^2 d)$ | $O(n^2)$ | 精确 |
| 稀疏注意力 | $O(n \sqrt{n} d)$ | $O(n \sqrt{n})$ | 近似 |
| 线性注意力 | $O(n d^2)$ | $O(nd)$ | 近似 |
| FlashAttention | $O(n^2 d)$ | $O(n)$ | 精确，IO 优化 |